# **Prerequisite Functions**

In [ ]:
# Connecting google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Downloading the dataset parts (9 subsets)

!wget --content-disposition "https://zenodo.org/records/3723295/files/subset2.zip?download=1"
!wget --content-disposition "https://zenodo.org/records/3723295/files/annotations.csv?download=1"

In [ ]:
# Unzipping the downlaoded files


folder_name = 'subset2'
scans_path = f'/content/{folder_name}.zip'
import zipfile

with zipfile.ZipFile(scans_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')


In [ ]:
!pip install SimpleITK

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 MB 10.8 MB/s eta 0:00:00


# **Implementation**

In [ ]:
# Loading a 3D scan using SimpleITK

import SimpleITK as sitk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def load_scan(path):
    itk_img = sitk.ReadImage(path)
    scan = sitk.GetArrayFromImage(itk_img)
    spacing = np.array(itk_img.GetSpacing())
    origin = np.array(itk_img.GetOrigin())
    transform_matrix = np.array(itk_img.GetDirection()).reshape(3, 3)  # Load the transform matrix
    return scan, spacing, origin, transform_matrix


In [ ]:
# Loading the annotations through the csv file

import pandas as pd

def load_annotations(annotations_path, available_seriesuids):
    annotations = []
    raw_annotations = pd.read_csv(annotations_path)
    for index, row in raw_annotations.iterrows():
        if row['seriesuid'] in available_seriesuids:
            annotations.append({
                'seriesuid': row['seriesuid'],
                'x': row['coordX'],
                'y': row['coordY'],
                'z': row['coordZ'],
                'diameter': row['diameter_mm'],
                'class': 1,
            })
    return annotations


In [ ]:
# Converting pyshical coordinates to voxel coordinates

import numpy as np


def physical_to_voxel_coords(coord, origin, spacing, transform_matrix):
    transformed_coord = np.dot(transform_matrix, coord - origin) + origin
    return np.round((transformed_coord - origin) / spacing).astype(int)

def convert_annotations_to_voxels_and_bboxes(annotations, scans_info):
    for annotation in annotations:
        seriesuid = annotation['seriesuid']
        if seriesuid in scans_info:
            origin = scans_info[seriesuid]['origin']
            spacing = scans_info[seriesuid]['spacing']
            transform_matrix = scans_info[seriesuid]['transform_matrix']
            scan_shape = scans_info[seriesuid]['scan'].shape
            coord = np.array([annotation['x'], annotation['y'], annotation['z']])
            voxel_coord = physical_to_voxel_coords(coord, origin, spacing, transform_matrix)
            diameter_voxel = annotation['diameter'] / spacing[0]

            half_diameter = diameter_voxel / 2
            bbox_x_min = max(0, voxel_coord[0] - half_diameter)
            bbox_y_min = max(0, voxel_coord[1] - half_diameter)
            bbox_x_max = min(scan_shape[2] - 1, voxel_coord[0] + half_diameter)
            bbox_y_max = min(scan_shape[1] - 1, voxel_coord[1] + half_diameter)

            if seriesuid == "1.3.6.1.4.1.14519.5.2.1.6279.6001.943403138251347598519939390311":
                print(f"Origin: {origin}, Spacing: {spacing}")
                print(f"Coordinates: {coord}, Voxel Coordinates: {voxel_coord}")
                print(f"Bounding Box: [{bbox_x_min}, {bbox_y_min}, {bbox_x_max}, {bbox_y_max}]")

            annotation.update({
                'voxelX': voxel_coord[0],
                'voxelY': voxel_coord[1],
                'voxelZ': voxel_coord[2],
                'bbox_x_top_left': int(bbox_x_min),
                'bbox_y_top_left': int(bbox_y_min),
                'bbox_x_bottom_right': int(bbox_x_max),
                'bbox_y_bottom_right': int(bbox_y_max)
            })

            del annotation['x']
            del annotation['y']
            del annotation['z']

    return annotations


In [ ]:
# Splitting the list of files into n batches

def split_list(lst, n):
    k, m = divmod(len(lst), n)
    return [lst[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n)]


In [ ]:
# Getting a list of the scans (".mhd" file paths)

import os

scans_path = []
for root, dirs, files in os.walk(f"/content/{folder_name}"):
      for file in files:
        file_path = os.path.join(root, file)
        file_name, file_extension = os.path.splitext(file_path)
        if file_extension == ".mhd":
          temp = os.path.join(root, file)
          scans_path.append(temp)


In [ ]:
# Batching the scans:
    # Getting the scans, spacing, and origin of each scan
    # You need to rerun this cell 4 times (Because the scans are divided in 4 batches due to resources limitation)
    # Sort the file paths

# Extract seriesuids from file paths
available_seriesuids = [os.path.splitext(os.path.basename(path))[0] for path in scans_path]
scans_path.sort()

# Splitting the scans into 4 batches
scan_batches = split_list(scans_path, 4)


# Path to the annotations CSV-s
annotations_path = '/content/annotations.csv'

# Load the merged annotations using load_annotations
annotations = load_annotations(annotations_path, available_seriesuids)

num_patch = 3
scans_info = {}
for path in scan_batches[num_patch]:
    scan, spacing, origin, transform_matrix = load_scan(path)
    seriesuid = os.path.splitext(os.path.basename(path))[0]  # Assuming filename is the seriesuid
    scans_info[seriesuid] = {
        'scan': scan,
        'spacing': spacing,
        'origin': origin,
        'transform_matrix': transform_matrix
    }

print(len(scans_info))


22


In [ ]:
# Convert annotations to voxel coordinates

annotations = convert_annotations_to_voxels_and_bboxes(annotations, scans_info)


Origin: [ 244.100006  241.600006 -310.75    ], Spacing: [0.87890601 0.87890601 1.25      ]
Coordinates: [-46.94966448  72.63645381 -95.64452131], Voxel Coordinates: [331 192 172]
Bounding Box: [315.3883062759904, 176.38830627599037, 346.6116937240096, 207.61169372400963]


In [ ]:
# Find the corresponding annotations for each slice


slice_annotations = {}

for seriesuid, info in scans_info.items():
    scan = info['scan']
    for z in range(scan.shape[0]):
        count = 0
        slice_key = f"{seriesuid}_slice_{z}"
        slice_annotations[slice_key] = []
        for index, annotation in enumerate(annotations):
            if annotation['seriesuid'] == seriesuid and annotation['voxelZ'] == z:
                count += 1
                slice_annotations[slice_key].append({
                    'bbox': [annotation['bbox_x_top_left'], annotation['bbox_y_top_left'], annotation['bbox_x_bottom_right'], annotation['bbox_y_bottom_right']],
                    'class': annotation['class'],
                })


In [ ]:
# Windowing of -1000 to 400
# 12-bit gray rescale


import numpy as np

def apply_windowing_12bit(slice, window_center, window_width):
    min_value = window_center - (window_width / 2)
    max_value = window_center + (window_width / 2)
    windowed_slice = np.clip(slice, min_value, max_value) # Performing windowing
    rescaled_slice = ((windowed_slice - min_value) / window_width) * 4095 # 12-bit gray scale
    return rescaled_slice

def window_scans_12bit(scans_info, window_center, window_width):
    windowed_scans_info = {}
    for seriesuid, info in scans_info.items():
        scan = info['scan']
        windowed_scan = np.zeros_like(scan, dtype=np.float32)  # Using float32 to maintain precision in the 0-1 range
        for z in range(scan.shape[0]):
            windowed_scan[z] = apply_windowing_12bit(scan[z], window_center, window_width)
        windowed_scans_info[seriesuid] = {
            'scan': windowed_scan,
        }
    return windowed_scans_info

window_center = (400 + (-1000)) / 2  # -300
window_width = 400 - (-1000)  # 1400

scans_info = window_scans_12bit(scans_info, window_center, window_width)


# **Saving images (png) & Annotations (json)**

In [ ]:
# Prerequisits for saving the annotations file (just due to "BoxMode.XYWH_ABS")

!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)
!pip install SimpleITK
!pip install imageio


In [ ]:
# Taking out the slices with nodules

slices_have_nodule = []

for key, value in slice_annotations.items():
  if len(value) > 0:
    temp = {}
    for item in value:
      temp[key] = [item]
    slices_have_nodule.append(temp)

len(slices_have_nodule)


20

In [ ]:
len(slices_have_nodule)

20

In [ ]:
# Saving the 12-bit gray scale images as tiff 16-bit (it's just a 16-bit storage for the 12-bit values) images

import numpy as np
import cv2
import imageio

def save_slice_as_tiff(slice_data, output_path):

    # ،he data is in the correct format (uint16)
    if slice_data.dtype != np.uint16:
        slice_data = slice_data.astype(np.uint16)

    # Save the image as 16-bit TIFF
    imageio.imwrite(output_path, slice_data)


In [ ]:
# Saving the slices as images


import matplotlib.pyplot as plt
import os
import numpy as np
from PIL import Image

# Define output directory for the slices
output_dir = "/content/drive/MyDrive/LUNA16/Nodule/Nodule_Images"
# os.makedirs(output_dir, exist_ok=True)

# Iterate through slices_have_nodule to save slices as images
for slice_info in slices_have_nodule:
    for slice_key, annotations in slice_info.items():
        seriesuid, slice_number = slice_key.rsplit('_slice_', 1)
        slice_number = int(slice_number)
        if seriesuid in scans_info:
            slice_data = scans_info[seriesuid]['scan'][slice_number]
            output_path = os.path.join(output_dir, f"{slice_key}.tiff")
            save_slice_as_tiff(slice_data, output_path)

print(f"Saved slices to {output_dir}")


Saved slices to /content/drive/MyDrive/LUNA16/Nodule/Nodule_Images


In [ ]:
# Saving annotations as a file (Based on Detectron2)

import json
from detectron2.structures import BoxMode


# Convert to Detectron2 format
def convert_annotations_to_detectron2(annotations):
    dataset_dicts = []
    for annotation in annotations:
        for file_name, anns in annotation.items():
            record = {
                "file_name": f"{file_name}.tiff",  # Assuming images are saved as PNG files
                "height": 512,
                "width": 512,
                "annotations": []
            }
            for ann in anns:
                bbox = ann['bbox']
                # Convert bbox from [xmin, ymin, xmax, ymax] to [xmin, ymin, width, height]
                width = bbox[2] - bbox[0]
                height = bbox[3] - bbox[1]
                bbox = [bbox[0], bbox[1], width, height]
                record["annotations"].append({
                    "bbox": bbox,
                    "bbox_mode": BoxMode.XYWH_ABS,
                    "category_id": ann['class']  # assuming class' index starts from 1
                })
            dataset_dicts.append(record)
    return dataset_dicts


# Convert annotations
detectron2_annotations = convert_annotations_to_detectron2(slices_have_nodule)

# Save to JSON file
output_json_path = f'/content/drive/MyDrive/LUNA16/Temp Annotations/LUNA16_{folder_name}_{num_patch}.json'
with open(output_json_path, 'w') as f:
    json.dump(detectron2_annotations, f)

print(f"Annotations saved to {output_json_path}")


Annotations saved to /content/drive/MyDrive/LUNA16/Temp Annotations/LUNA16_subset2_3.json
